### Import libraries

In [1]:
import pandas as pd
import numpy as np
import os

### Import data

In [2]:
# Get the current working directory
current_directory = os.getcwd() 
#print(current_directory)

In [5]:
df = pd.read_excel("casinoactivitydata.xlsx")
df.head()

,time_eet,user_id,customer_dim_id,market_code,casino_game_name,game_type,is_live,provider,provider_id,bets_eur,wins_eur,bet_count,avg_bet_size
0,2025-09-01 00:00:00,H1N8M5R7,10001.0,EE,Book of Dead,slot,False,Play'n GO,5,15.0,0.0,15,1.0
1,2025-09-01 01:00:00,J2S9O6Q8,10002.0,CL,Starburst,slot,False,NetEnt,1,40.0,5.0,10,4.0
2,2025-09-01 02:00:00,K3T0P7R9,10003.0,SE,Lightning Roulette,Lightning Roulette,True,Evolution,101,150.0,300.0,5,30.0
3,2025-09-01 03:00:00,L4U1Q8S0,10004.0,FI,Money Train 3,slot,False,Red Tiger,4,25.0,2.5,5,5.0
4,2025-09-01 04:00:00,M5V2R9T1,10005.0,EE,Gonzo's Quest,slot,False,NetEnt,1,120.0,10.0,20,6.0


### Data Exploration

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 720 entries, 0 to 719
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   time_eet          720 non-null    datetime64[ns]
 1   user_id           720 non-null    object        
 2   customer_dim_id   500 non-null    float64       
 3   market_code       720 non-null    object        
 4   casino_game_name  720 non-null    object        
 5   game_type         720 non-null    object        
 6   is_live           720 non-null    bool          
 7   provider          720 non-null    object        
 8   provider_id       720 non-null    int64         
 9   bets_eur          720 non-null    float64       
 10  wins_eur          720 non-null    float64       
 11  bet_count         720 non-null    int64         
 12  avg_bet_size      720 non-null    float64       
dtypes: bool(1), datetime64[ns](1), float64(4), int64(2), object(5)
memory usage: 68.

The dataset is made of 13 variables and 720 rows. It contains grouped data for every hour of every day in September 2025, about four different markets (Chile, Estonia, Finland and Sweden) with bets on different games and info about the bets and their outcomes. 

The variable *customer_dim_id* seems to be a sort of index starting at 10001 and increasing by one every hour, ordered by the field *time_eet* (which contains the timestamp information), but it's not clean as it stops at 10500, so it needs to be cleaned to be useful. 

The variables *provider*, *provider_id*, *game_type*, *casino_game_name* and *is_live* are all categorical variables describing characteristics about the games played by the customers. They need some data cleaning operations: 
- *provider* and *provider_id* aren't always consistent (but looking at *provider* with *game_type* it seems that *provider* is the correct one);
- in *game_type* I correct it to be always with uppercase letter at the beginning;
- the variable *casino_game_name* also needs some cleaning operations, as the game Starburst is written in 4 different ways.

The numerical variables seems to be all correct, except for the *avg_bet_size*, not always as *bets_eur*/*bet_count*, so I need to correct that result. 

In [7]:
df.describe()

,time_eet,customer_dim_id,provider_id,bets_eur,wins_eur,bet_count,avg_bet_size
count,720,500.000000,720.000000,720.000000,720.000000,720.000000,720.000000
mean,2025-09-15 23:30:00,10250.500000,16.043056,70.395833,62.050792,7.461111,19.883014
min,2025-09-01 00:00:00,10001.000000,1.000000,4.000000,0.000000,1.000000,0.250000
25%,2025-09-08 11:45:00,10125.750000,2.000000,15.000000,2.000000,3.000000,1.000000
50%,2025-09-15 23:30:00,10250.500000,5.000000,25.000000,5.000000,5.000000,5.000000
75%,2025-09-23 11:15:00,10375.250000,5.000000,50.000000,50.000000,10.000000,25.000000
max,2025-09-30 23:00:00,10500.000000,101.000000,1250.000000,1250.000000,60.000000,250.000000
std,NaN,144.481833,32.586286,125.016559,143.430924,6.761680,34.873558


In [11]:
df[['provider', 'game_type','casino_game_name','is_live']].value_counts().reset_index(name='count')

,provider,game_type,casino_game_name,is_live,count
0,Evolution,Baccarat Squeeze,Baccarat Squeeze,True,38
1,Evolution,Crazy Time,Crazy Time,True,38
2,Evolution,Infinite Blackjack,Infinite Blackjack,True,38
3,Evolution,Lightning Roulette,Lightning Roulette,True,38
4,Microgaming,slot,Mega Moolah,False,38
5,NetEnt,slot,Gonzo's Quest,False,38
6,Pragmatic Play,slot,The Dog House,False,38
7,Play'n GO,slot,Legacy of Dead,False,38
8,Play'n GO,slot,Fire Joker,False,38
9,Play'n GO,slot,Book of Dead,False,38


In [13]:
df[['provider', 'game_type']].value_counts().reset_index(name='count')

,provider,game_type,count
0,Play'n GO,slot,228
1,NetEnt,slot,76
2,Pragmatic Play,slot,76
3,Red Tiger,slot,75
4,Evolution,Infinite Blackjack,38
5,Evolution,Baccarat Squeeze,38
6,Evolution,Crazy Time,38
7,Evolution,Lightning Roulette,38
8,Microgaming,slot,38
9,Push Gaming,slot,38


In [34]:
df[['provider', 'game_type', 'provider_id']].value_counts().reset_index(name='count')

,provider,game_type,provider_id,count
0,Play'n GO,slot,5,228
1,NetEnt,slot,1,76
2,Pragmatic Play,slot,6,76
3,Red Tiger,slot,4,75
4,Push Gaming,slot,3,38
5,Microgaming,slot,2,38
6,Evolution,Baccarat Squeeze,1,21
7,Evolution,Crazy Time,1,20
8,Evolution,Infinite Blackjack,1,19
9,Evolution,Infinite Blackjack,101,19


### Data cleaning

In [47]:
df_clean = df

In [48]:
df_clean = df_clean.sort_values(by='time_eet').reset_index(drop=True)
df_clean['customer_dim_id'] = range(10001, 10001 + len(df_clean))  # clean customer_dim_id based on time_eet ordering

In [49]:
df_clean['casino_game_name_clean'] = df_clean['casino_game_name'].str.title()    # clean upper/lowercase issues


df_clean['casino_game_name_clean'] = df_clean['casino_game_name_clean'].replace(
    {'Star Burst Slot': 'Starburst'}    # replace Star Burst Slot with Starburst
)

In [50]:
market_mapping = {
    'EE': 'Estonia',
    'CL': 'Chile',
    'SE': 'Sweden',
    'FI': 'Finland'
}
df_clean['market_full_name'] = df_clean['market_code'].map(market_mapping)    # add full market names

In [51]:
df_clean['avg_bet_size_clean'] = np.where(
    df_clean['bet_count'] > 0,
    df_clean['bets_eur'] / df_clean['bet_count'],
    0.0
)

In [ ]:
# remap provider_id based on provider
provider_id_mapping = {
    'NetEnt': 1,
    'Microgaming': 2,
    'Push Gaming': 3,
    'Red Tiger': 4,
    "Play'n GO": 5,
    'Pragmatic Play': 6,
    'Evolution': 101
}
df_clean['provider_id_clean'] = df_clean['provider'].map(provider_id_mapping)

### Final version

In [54]:
df_clean.head()

,time_eet,user_id,customer_dim_id,market_code,casino_game_name,game_type,is_live,provider,provider_id,bets_eur,wins_eur,bet_count,avg_bet_size,casino_game_name_clean,market_full_name,avg_bet_size_clean,provider_id_clean
0,2025-09-01 00:00:00,H1N8M5R7,10001,EE,Book of Dead,slot,False,Play'n GO,5,15.0,0.0,15,1.0,Book Of Dead,Estonia,1.0,5
1,2025-09-01 01:00:00,J2S9O6Q8,10002,CL,Starburst,slot,False,NetEnt,1,40.0,5.0,10,4.0,Starburst,Chile,4.0,1
2,2025-09-01 02:00:00,K3T0P7R9,10003,SE,Lightning Roulette,Lightning Roulette,True,Evolution,101,150.0,300.0,5,30.0,Lightning Roulette,Sweden,30.0,101
3,2025-09-01 03:00:00,L4U1Q8S0,10004,FI,Money Train 3,slot,False,Red Tiger,4,25.0,2.5,5,5.0,Money Train 3,Finland,5.0,4
4,2025-09-01 04:00:00,M5V2R9T1,10005,EE,Gonzo's Quest,slot,False,NetEnt,1,120.0,10.0,20,6.0,Gonzo'S Quest,Estonia,6.0,1


In [56]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 720 entries, 0 to 719
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   time_eet                720 non-null    datetime64[ns]
 1   user_id                 720 non-null    object        
 2   customer_dim_id         720 non-null    int64         
 3   market_code             720 non-null    object        
 4   casino_game_name        720 non-null    object        
 5   game_type               720 non-null    object        
 6   is_live                 720 non-null    bool          
 7   provider                720 non-null    object        
 8   provider_id             720 non-null    int64         
 9   bets_eur                720 non-null    float64       
 10  wins_eur                720 non-null    float64       
 11  bet_count               720 non-null    int64         
 12  avg_bet_size            720 non-null    float64   

In [55]:
df_clean.describe()

,time_eet,customer_dim_id,provider_id,bets_eur,wins_eur,bet_count,avg_bet_size,avg_bet_size_clean,provider_id_clean
count,720,720.000000,720.000000,720.000000,720.000000,720.000000,720.000000,720.000000,720.000000
mean,2025-09-15 23:30:00,10360.500000,16.043056,70.395833,62.050792,7.461111,19.883014,19.883009,29.515278
min,2025-09-01 00:00:00,10001.000000,1.000000,4.000000,0.000000,1.000000,0.250000,0.250000,1.000000
25%,2025-09-08 11:45:00,10180.750000,2.000000,15.000000,2.000000,3.000000,1.000000,1.000000,4.000000
50%,2025-09-15 23:30:00,10360.500000,5.000000,25.000000,5.000000,5.000000,5.000000,5.000000,5.000000
75%,2025-09-23 11:15:00,10540.250000,5.000000,50.000000,50.000000,10.000000,25.000000,25.000000,101.000000
max,2025-09-30 23:00:00,10720.000000,101.000000,1250.000000,1250.000000,60.000000,250.000000,250.000000,101.000000
std,NaN,207.990384,32.586286,125.016559,143.430924,6.761680,34.873558,34.873561,42.700331


In [ ]:
df_clean.to_excel('casinoactivitydata_cleaned.xlsx', index=False)   # export cleaned data to a new Excel file for the Tableau dashboard